## Métricas de evaluación cuantitativas

In [1]:
import numpy as np

def mse_trajectories(generated, target):
    """
    Mean Squared Error entre dos trayectorias.
    Ambas deben ser arrays de shape (N, 2).
    """
    return np.mean((generated - target) ** 2)

def chamfer_distance(traj_a, traj_b):
    """
    Chamfer Distance entre dos conjuntos de puntos 2D.
    Para cada punto en A, busca el más cercano en B y viceversa.
    """
    from scipy.spatial.distance import cdist
    dist_matrix = cdist(traj_a, traj_b)
    # Para cada punto en A, la distancia mínima a B
    min_a_to_b = np.mean(np.min(dist_matrix, axis=1))
    # Para cada punto en B, la distancia mínima a A
    min_b_to_a = np.mean(np.min(dist_matrix, axis=0))
    return (min_a_to_b + min_b_to_a) / 2.0

def smoothness(trajectory):
    """
    Mide la suavidad de una trayectoria como la magnitud promedio
    de la segunda derivada discreta (aceleración / jerk).
    Valores más bajos = trayectoria más suave.
    """
    if len(trajectory) < 3:
        return 0.0
    # Primera derivada (velocidad)
    velocity = np.diff(trajectory, axis=0)
    # Segunda derivada (aceleración)
    acceleration = np.diff(velocity, axis=0)
    # Magnitud promedio de la aceleración
    return np.mean(np.linalg.norm(acceleration, axis=1))

def evaluate_model(model, scheduler, test_graphs, device='cpu', num_samples=10):
    """
    Evalúa el modelo sobre un subconjunto de grafos de test.
    Retorna las métricas promedio.
    """
    mse_scores = []
    chamfer_scores = []
    smoothness_scores = []

    for i, graph in enumerate(test_graphs[:num_samples]):
        # Generar trayectoria
        generated, _ = generate_trajectory(model, scheduler, graph, device=device)
        generated_np = generated.numpy()

        # Target (ground truth, ya normalizado dentro del grafo)
        target_np = graph['action'].x.numpy()

        # Calcular métricas
        mse_scores.append(mse_trajectories(generated_np, target_np))
        chamfer_scores.append(chamfer_distance(generated_np, target_np))
        smoothness_scores.append(smoothness(generated_np))

    results = {
        'MSE (promedio)': np.mean(mse_scores),
        'Chamfer Distance (promedio)': np.mean(chamfer_scores),
        'Smoothness (promedio)': np.mean(smoothness_scores),
    }

    print("=" * 50)
    print("RESULTADOS DE EVALUACIÓN")
    print("=" * 50)
    for metric, value in results.items():
        print(f"  {metric}: {value:.6f}")
    print("=" * 50)

    return results


In [ ]:
import torch
from src.classes.DDPMScheduler import DDPMScheduler

In [ ]:
def load_checkpoint(path, model, optimizer=None, device='cpu'):
    """Carga el estado del modelo desde un checkpoint."""
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    train_loss = checkpoint['train_loss']
    val_loss = checkpoint.get('val_loss', None)
    print(f"✅ Checkpoint cargado desde {path} (epoch {epoch})")
    return epoch, train_loss, val_loss